In [2]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Load Olist Datasets
# ---------------------------------------------------------
print("Loading Olist datasets...")
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
geo = pd.read_csv('olist_geolocation_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

# Merge translation for readable category names
products = products.merge(translation, on='product_category_name', how='left')
products['category_name'] = products['product_category_name_english'].fillna(products['product_category_name']).fillna('Unknown')

# ---------------------------------------------------------
# 2. Process Geolocation Data (Average Lat/Lng per Zip Code)
# ---------------------------------------------------------
geo_coords = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

customers = customers.merge(
    geo_coords,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)
sellers = sellers.merge(
    geo_coords,
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

# Filter sellers with valid coordinates
valid_sellers = sellers.dropna(subset=['geolocation_lat', 'geolocation_lng']).copy()

# ---------------------------------------------------------
# 3. Distance Calculation Helper (Haversine Formula)
# ---------------------------------------------------------
def calculate_distance_km(lat1, lon1, lat2, lon2):
    """Calculates distance between coordinates in kilometers."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 6367 * 2 * np.arcsin(np.sqrt(a))

def get_top_5_nearest_sellers(cust_lat, cust_lng, seller_df=valid_sellers, product_id=None):
    """Finds top 5 nearest sellers, optionally filtering by inventory / seller who sold the product."""
    df = seller_df.copy()
    if product_id is not None:
        sellers_with_product = order_items[order_items['product_id'] == product_id]['seller_id'].unique()
        if len(sellers_with_product) > 0:
            df = df[df['seller_id'].isin(sellers_with_product)]

    if pd.isna(cust_lat) or pd.isna(cust_lng) or len(df) == 0:
        return "N/A", "N/A"

    distances = calculate_distance_km(cust_lat, cust_lng, df['geolocation_lat'].values, df['geolocation_lng'].values)
    df['distance_km'] = distances
    top5 = df.sort_values('distance_km').head(5)

    seller_ids = ", ".join(top5['seller_id'].tolist())
    seller_details = ", ".join([f"{sid} ({city}, {state} - {dist:.1f}km)" for sid, city, state, dist in zip(
        top5['seller_id'], top5['seller_city'], top5['seller_state'], top5['distance_km']
    )])
    return seller_ids, seller_details

# ---------------------------------------------------------
# 4. Product Recommendation Logic
# ---------------------------------------------------------
# Build co-occurrence matrix / frequent category recommendations
merged_orders = order_items.merge(orders[['order_id', 'customer_id']], on='order_id')
merged_orders = merged_orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
merged_orders = merged_orders.merge(products[['product_id', 'category_name']], on='product_id')

# Top popular products overall (fallback recommendation)
top_popular_products = order_items['product_id'].value_counts().head(5).index.tolist()

def get_recommended_items_for_customer(cust_unique_id):
    """Recommends top 5 products based on customer purchase category history."""
    user_purchases = merged_orders[merged_orders['customer_unique_id'] == cust_unique_id]

    if len(user_purchases) > 0:
        user_categories = user_purchases['category_name'].unique()
        recs = products[products['category_name'].isin(user_categories) & ~products['product_id'].isin(user_purchases['product_id'])]
        rec_ids = recs['product_id'].head(5).tolist()
    else:
        rec_ids = []

    if len(rec_ids) < 5:
        # Fill remaining with popular items
        for p_id in top_popular_products:
            if p_id not in rec_ids and len(rec_ids) < 5:
                rec_ids.append(p_id)

    rec_products = products[products['product_id'].isin(rec_ids)]
    rec_names = [f"{pid} ({cat})" for pid, cat in zip(rec_products['product_id'], rec_products['category_name'])]

    return ", ".join(rec_ids), " | ".join(rec_names)

def get_recommended_items_for_product(product_id):
    """Recommends top 5 similar items in the same product category for new products."""
    prod_row = products[products['product_id'] == product_id]
    if len(prod_row) > 0:
        cat = prod_row.iloc[0]['category_name']
        recs = products[(products['category_name'] == cat) & (products['product_id'] != product_id)].head(5)
        rec_ids = recs['product_id'].tolist()
    else:
        rec_ids = top_popular_products[:5]

    rec_products = products[products['product_id'].isin(rec_ids)]
    rec_names = [f"{pid} ({cat})" for pid, cat in zip(rec_products['product_id'], rec_products['category_name'])]

    return ", ".join(rec_ids), " | ".join(rec_names)

# ---------------------------------------------------------
# 5. Process Batch Customer & Purchased Data
# ---------------------------------------------------------
print("Processing Customer Recommendations and Nearest Sellers...")
sample_customers = customers.head(100).copy() # Processing first 100 sample customers

records = []
for idx, row in sample_customers.iterrows():
    c_id = row['customer_id']
    u_id = row['customer_unique_id']
    lat = row['geolocation_lat']
    lng = row['geolocation_lng']

    # Purchased products
    p_items = merged_orders[merged_orders['customer_id'] == c_id]['product_id'].tolist()
    purchased_str = ", ".join(p_items) if p_items else "None"

    # Nearest 5 Sellers
    top5_seller_ids, top5_seller_names = get_top_5_nearest_sellers(lat, lng)

    # Recommended 5 Items
    rec_item_ids, rec_item_names = get_recommended_items_for_customer(u_id)

    records.append({
        'Customer ID': c_id,
        'Customer Unique ID': u_id,
        'Customer City': row['customer_city'],
        'Customer State': row['customer_state'],
        'Purchased Product IDs': purchased_str,
        'Top 5 Nearest Seller IDs': top5_seller_ids,
        'Top 5 Nearest Seller Names & Details': top5_seller_names,
        'Recommended Product IDs': rec_item_ids,
        'Recommended Product Names & Categories': rec_item_names
    })

df_deployed = pd.DataFrame(records)

# ---------------------------------------------------------
# 6. Interactive Tab Entry Functionality for New Products
# ---------------------------------------------------------
def predict_new_product_recommendations(new_product_id, reference_customer_id=None):
    """
    Automated prediction for upcoming/new product entry.
    Predicts nearest seller stocking/selling the item and recommended complement items.
    """
    if reference_customer_id:
        cust_row = customers[customers['customer_id'] == reference_customer_id]
        if len(cust_row) > 0:
            lat = cust_row.iloc[0]['geolocation_lat']
            lng = cust_row.iloc[0]['geolocation_lng']
        else:
            lat, lng = np.nan, np.nan
    else:
        # Default center lat/lng if no customer ID passed
        lat, lng = geo_coords['geolocation_lat'].mean(), geo_coords['geolocation_lng'].mean()

    seller_ids, seller_names = get_top_5_nearest_sellers(lat, lng, product_id=new_product_id)
    rec_ids, rec_names = get_recommended_items_for_product(new_product_id)

    return {
        'New / Upcoming Product ID': new_product_id,
        'Reference Customer ID': reference_customer_id if reference_customer_id else 'All Customers',
        'Top 5 Nearest Seller IDs': seller_ids,
        'Top 5 Nearest Seller Names & Details': seller_names,
        'Recommended Product IDs': rec_ids,
        'Recommended Product Names & Categories': rec_names
    }

# Sample entry for upcoming products tab
sample_new_product = products['product_id'].iloc[0]
sample_cust_id = sample_customers['customer_id'].iloc[0]

new_product_entry = predict_new_product_recommendations(sample_new_product, sample_cust_id)
df_new_product_tab = pd.DataFrame([new_product_entry])

# ---------------------------------------------------------
# 7. Generate Excel Workbook
# ---------------------------------------------------------
output_file = 'Customer_Seller_Recommendations.xlsx'
print(f"Saving workbook to {output_file}...")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_deployed.to_excel(writer, sheet_name='Deployed_Purchased_Items', index=False)
    df_new_product_tab.to_excel(writer, sheet_name='Enter_New_Product_Tab', index=False)

print("Excel generation complete! Download 'Customer_Seller_Recommendations.xlsx' directly in Colab.")

Loading Olist datasets...
Processing Customer Recommendations and Nearest Sellers...
Saving workbook to Customer_Seller_Recommendations.xlsx...
Excel generation complete! Download 'Customer_Seller_Recommendations.xlsx' directly in Colab.


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create input widgets for Colab UI
prod_input = widgets.Text(
    value=products['product_id'].iloc[0],
    description='Product ID:',
    layout=widgets.Layout(width='450px')
)

cust_input = widgets.Text(
    value=customers['customer_id'].iloc[0],
    description='Customer ID:',
    layout=widgets.Layout(width='450px')
)

button = widgets.Button(description="Predict Nearest Sellers & Packages", button_style='primary')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        p_id = prod_input.value.strip()
        c_id = cust_input.value.strip()

        # Calculate
        res = predict_new_product_recommendations(p_id, c_id)

        print("="*60)
        print(f"PREDICTION RESULTS FOR PRODUCT: {p_id}")
        print("="*60)
        print(f"\n📍 TOP 5 NEAREST SELLER IDs:\n{res['Top 5 Nearest Seller IDs']}")
        print(f"\n🚚 SELLER DETAILS & DISTANCES:\n{res['Top 5 Nearest Seller Names & Details']}")
        print(f"\n📦 RECOMMENDED COMPLEMENTARY PRODUCT IDs:\n{res['Recommended Product IDs']}")
        print(f"\n🏷️ RECOMMENDED PRODUCTS & CATEGORIES:\n{res['Recommended Product Names & Categories']}")

button.on_click(on_button_clicked)

display(prod_input, cust_input, button, output)

Text(value='1e9e8ef04dbcff4541ed26657ea517e5', description='Product ID:', layout=Layout(width='450px'))

Text(value='06b8999e2fba1a1fbc88172c00ba8bc7', description='Customer ID:', layout=Layout(width='450px'))

Button(button_style='primary', description='Predict Nearest Sellers & Packages', style=ButtonStyle())

Output()